In [1]:
%run ../../Utils/yp_utils.py

# Initial setup

In [2]:
paper_pmid = 38287338
paper_name = 'strawn_mayor_2024' 

In [3]:
datasets = pd.read_csv('extras/YeastPhenome_' + str(paper_pmid) + '_datasets_list.txt', sep='\t', header=None, names=['dataset_id', 'name'])

In [4]:
datasets.set_index('dataset_id', inplace=True)

# Load & process the data

In [5]:
original_data = pd.read_excel('raw_data/12934_2024_2298_MOESM1_ESM.xlsx', 
                              sheet_name='Full Data', header=1)

In [6]:
print('Original data dimensions: %d x %d' % (original_data.shape))

Original data dimensions: 6145 x 53


In [7]:
original_data.head()

,ORF,Gene Name,"Plate,Rep,Row,Col,Zone,Growth_ID",Zone,Growth (1 vs 0),Max_Intensity,Min_Intensity,Integrated_Intensity,Mean_Intensity,Median_Intensity,...,Location_MaxIntensity_X.2,Location_MaxIntensity_Y.2,NormMeanIntensity.2,ZoneCorrect (Y/N).2,ZoneCorrectedNormMeanIntensity.2,MAD Z scores.2,ABTSOverlayHit.2,#Hits/3,ORF.1,Gene Name.1
0,undefined,undefined,101010110.0,1.0,0.0,38.169902,11.613484,198289.612168,24.261546,23.778204,...,195.841841,166.369116,NaN,1.0,NaN,NaN,NaN,0,undefined,undefined
1,YLL040C,VPS13,101010211.0,1.0,1.0,75.447475,54.112836,544274.384054,66.594198,67.447475,...,368.955598,163.319241,1.175031,1.0,1.078266,2.355861,NaN,2,YLL040C,VPS13
2,YAL068C,YAL068C,101010311.0,1.0,1.0,70.669269,46.335922,498296.037715,60.968560,62.196495,...,545.643607,163.365216,1.097844,1.0,1.007435,0.559600,NaN,0,YAL068C,YAL068C
3,undefined,undefined,101010410.0,1.0,0.0,40.335922,21.167311,257398.835379,31.493801,31.833977,...,725.538759,164.756904,NaN,1.0,NaN,NaN,NaN,0,undefined,undefined
4,YAL067C,SEO1,101010511.0,1.0,1.0,70.444884,50.390411,499124.645496,61.069943,61.556423,...,902.750521,163.042344,1.074729,1.0,0.986223,0.021669,NaN,0,YAL067C,SEO1


In [8]:
original_data['orf'] = original_data['ORF.1'].astype(str)

In [9]:
# Eliminate all white spaces & capitalize
original_data['orf'] = clean_orf(original_data['orf'])

In [10]:
# Translate to ORFs 
original_data['orf'] = translate_sc(original_data['orf'], to='orf')

In [11]:
# Make sure everything translated ok
t = looks_like_orf(original_data['orf'])
print(original_data.loc[~t,'orf'].unique())

['UNDEFINED' 'NAN']


In [12]:
original_data = original_data.loc[t,]

In [13]:
original_data['data'] = original_data[['MAD Z scores','MAD Z scores.1', 'MAD Z scores.2']].mean(axis=1)

In [14]:
original_data.set_index('orf', inplace=True)

In [15]:
original_data = original_data[['data']].copy()

In [16]:
original_data = original_data.groupby(original_data.index).mean()

In [17]:
original_data.shape

(4765, 1)

In [18]:
original_data.sort_values(by='data', ascending=False).head()

,data
orf,
YKL073W,6.405864
YCL008C,5.886256
YOR089C,5.143384
YJL062W,4.883972
YLR119W,4.815076


# Prepare the final dataset

In [19]:
data = original_data.copy()

In [20]:
dataset_ids = [22270]
datasets = datasets.reindex(index=dataset_ids)

In [21]:
lst = [datasets.index.values, ['value']*datasets.shape[0]]
tuples = list(zip(*lst))
idx = pd.MultiIndex.from_tuples(tuples, names=['dataset_id','data_type'])
data.columns = idx

In [22]:
data.head()

dataset_id,22270
data_type,value
orf,
YAL002W,3.069646
YAL004W,1.847721
YAL005C,0.669737
YAL007C,0.573501
YAL008W,0.367687


## Subset to the genes currently in SGD

In [23]:
genes = pd.read_csv(path_to_genes, sep='\t', index_col='id')
genes = genes.reset_index().set_index('systematic_name')
gene_ids = genes.reindex(index=data.index.values)['id'].values
num_missing = np.sum(np.isnan(gene_ids))
print('ORFs missing from SGD: %d' % num_missing)

ORFs missing from SGD: 22


In [24]:
data['gene_id'] = gene_ids
data = data.loc[data['gene_id'].notnull()]
data['gene_id'] = data['gene_id'].astype(int)
data = data.reset_index().set_index(['gene_id','orf'])

data.head()

,dataset_id,22270
,data_type,value
gene_id,orf,
2,YAL002W,3.069646
1863,YAL004W,1.847721
4,YAL005C,0.669737
5,YAL007C,0.573501
6,YAL008W,0.367687


# Normalize

In [25]:
data_norm = normalize_phenotypic_scores(data, has_tested=True)

In [26]:
# Assign proper column names
lst = [datasets.index.values, ['valuez']*datasets.shape[0]]
tuples = list(zip(*lst))
idx = pd.MultiIndex.from_tuples(tuples, names=['dataset_id','data_type'])
data_norm.columns = idx

In [27]:
data_norm[data.isnull()] = np.nan
data_all = data.join(data_norm)

data_all.head()

dataset_id          22270          
data_type           value    valuez
gene_id orf                        
2       YAL002W  3.069646  2.554376
1863    YAL004W  1.847721  1.539915
4       YAL005C  0.669737  0.561936
5       YAL007C  0.573501  0.482040
6       YAL008W  0.367687  0.311170

# Print out

In [28]:
for f in ['value','valuez']:
    df = data_all.xs(f, level='data_type', axis=1).copy()
    df.columns = datasets['name'].values
    df = df.droplevel('gene_id', axis=0)
    df.to_csv(paper_name + '_' + f + '.txt', sep='\t')